In [0]:
storage_account = "sreyaastorage"

silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/"

print(silver_path)

abfss://silver@sreyaastorage.dfs.core.windows.net/


In [0]:

storage_account = "sreyaastorage"
host = f"{storage_account}.dfs.core.windows.net"

client_id = dbutils.secrets.get(
    scope="sreyascope",
    key="clientid"
)
client_secret = dbutils.secrets.get(
    scope="sreyascope",
    key="clientsecretvalue"
)
tenant_id = dbutils.secrets.get(
    scope="sreyascope",
    key="tenantsecretid"
)

spark.conf.set(
    f"fs.azure.account.auth.type.{host}", "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{host}",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{host}", client_id
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{host}", client_secret
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{host}",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

print("OAuth configuration applied")


OAuth configuration applied


In [0]:

silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/"

transactions_silver = spark.read.parquet(
    f"{silver_path}transactions/"
)

products_silver = spark.read.parquet(
    f"{silver_path}products/"
)

stores_silver = spark.read.parquet(
    f"{silver_path}stores/"
)

customers_silver = spark.read.parquet(
    f"{silver_path}customers/"
)

print("Silver data loaded successfully!")


Silver data loaded successfully!


In [0]:

from pyspark.sql.functions import col, sum as spark_sum

# Join transactions with products
sales_details = transactions_silver.join(
    products_silver,
    on="product_id",
    how="inner"
)

# Calculate revenue for each transaction
sales_details = sales_details.withColumn(
    "revenue",
    col("quantity") * col("price")
)

display(sales_details)


product_id,transaction_id,customer_id,store_id,quantity,transaction_date,product_name,category,price,revenue
7,6,110,3,5,2025-01-04,Smartwatch,Electronics,4999.00,24995.00
8,19,116,4,4,2024-07-14,Desk Organizer,Accessories,399.00,1596.00
6,16,118,2,4,2024-11-29,Water Bottle,Fitness,299.00,1196.00
8,17,109,5,5,2024-07-10,Desk Organizer,Accessories,399.00,1995.00
5,12,108,1,4,2025-05-26,Notebook Set,Stationery,149.00,596.00
5,5,105,2,1,2025-03-17,Notebook Set,Stationery,149.00,149.00
9,22,107,3,4,2024-11-16,Dumbbell Set,Fitness,1999.00,7996.00
1,24,125,5,1,2024-07-14,Wireless Mouse,Electronics,799.99,799.99
1,11,102,3,2,2024-08-11,Wireless Mouse,Electronics,799.99,1599.98
5,20,130,1,2,2024-07-30,Notebook Set,Stationery,149.00,298.00


In [0]:

display(transactions_silver)


transaction_id,customer_id,product_id,store_id,quantity,transaction_date
6,110,7,3,5,2025-01-04
19,116,8,4,4,2024-07-14
16,118,6,2,4,2024-11-29
17,109,8,5,5,2024-07-10
12,108,5,1,4,2025-05-26
5,105,5,2,1,2025-03-17
22,107,9,3,4,2024-11-16
24,125,1,5,1,2024-07-14
11,102,1,3,2,2024-08-11
20,130,5,1,2,2024-07-30


In [0]:
from pyspark.sql.functions import col, sum as spark_sum

sales_details = transactions_silver.join(
    products_silver,
    on="product_id",
    how="inner"
).withColumn(
    "revenue",
    col("quantity") * col("price")
)

display(sales_details)

product_id,transaction_id,customer_id,store_id,quantity,transaction_date,product_name,category,price,revenue
7,6,110,3,5,2025-01-04,Smartwatch,Electronics,4999.00,24995.00
8,19,116,4,4,2024-07-14,Desk Organizer,Accessories,399.00,1596.00
6,16,118,2,4,2024-11-29,Water Bottle,Fitness,299.00,1196.00
8,17,109,5,5,2024-07-10,Desk Organizer,Accessories,399.00,1995.00
5,12,108,1,4,2025-05-26,Notebook Set,Stationery,149.00,596.00
5,5,105,2,1,2025-03-17,Notebook Set,Stationery,149.00,149.00
9,22,107,3,4,2024-11-16,Dumbbell Set,Fitness,1999.00,7996.00
1,24,125,5,1,2024-07-14,Wireless Mouse,Electronics,799.99,799.99
1,11,102,3,2,2024-08-11,Wireless Mouse,Electronics,799.99,1599.98
5,20,130,1,2,2024-07-30,Notebook Set,Stationery,149.00,298.00


In [0]:
gold_product_sales = sales_details.groupBy(
    "product_id"
).agg(
    spark_sum("quantity").alias("total_quantity_sold"),
    spark_sum("revenue").alias("total_revenue")
)

display(gold_product_sales)

product_id,total_quantity_sold,total_revenue
1,18,14399.82
6,11,3289.00
3,9,4491.00
5,7,1043.00
9,8,15992.00
8,25,9975.00
7,15,74985.00
2,8,10395.92


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS sreyadatabricks.gold
""")

gold_product_sales.write \
    .mode("overwrite") \
    .saveAsTable("sreyadatabricks.gold.product_sales")

print("Gold product sales table saved!")

Gold product sales table saved!


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS sreyadatabricks.gold
""")

gold_product_sales.write \
    .mode("overwrite") \
    .saveAsTable("sreyadatabricks.gold.product_sales")

print("Gold product sales table saved successfully!")

Gold product sales table saved successfully!


In [0]:
display(spark.table("sreyadatabricks.gold.product_sales"))

product_id,total_quantity_sold,total_revenue
1,18,14399.82
6,11,3289.00
3,9,4491.00
5,7,1043.00
9,8,15992.00
8,25,9975.00
7,15,74985.00
2,8,10395.92


In [0]:

from pyspark.sql.functions import col, sum as spark_sum

# Join transactions with store details
store_sales = transactions_silver.join(
    stores_silver,
    on="store_id",
    how="inner"
).join(
    products_silver.select("product_id", "price"),
    on="product_id",
    how="inner"
).withColumn(
    "revenue",
    col("quantity") * col("price")
)

# Summarize sales by store
gold_store_sales = store_sales.groupBy(
    "store_id"
).agg(
    spark_sum("quantity").alias("total_quantity_sold"),
    spark_sum("revenue").alias("total_revenue")
)

display(gold_store_sales)


store_id,total_quantity_sold,total_revenue
1,8,1692.00
3,22,42186.91
5,16,29284.99
4,35,24072.92
2,20,37333.92


In [0]:

gold_store_sales.write \
    .mode("overwrite") \
    .saveAsTable("sreyadatabricks.gold.store_sales")

print("Store sales table saved!")


Store sales table saved!


In [0]:

from pyspark.sql.functions import date_format

monthly_sales = sales_details.withColumn(
    "sales_month",
    date_format(col("transaction_date"), "yyyy-MM")
)

gold_monthly_sales = monthly_sales.groupBy(
    "sales_month"
).agg(
    spark_sum("quantity").alias("total_quantity_sold"),
    spark_sum("revenue").alias("total_revenue")
).orderBy("sales_month")

display(gold_monthly_sales)


sales_month,total_quantity_sold,total_revenue
2024-07,17,8688.94
2024-08,7,8097.43
2024-09,5,2997.97
2024-10,8,5998.93
2024-11,17,27083.00
2024-12,5,1995.00
2025-01,10,49990.00
2025-03,10,3740.00
2025-04,2,3998.00
2025-05,16,7985.47


In [0]:

gold_monthly_sales.write \
    .mode("overwrite") \
    .saveAsTable("sreyadatabricks.gold.monthly_sales")

print("Monthly sales table saved!")


Monthly sales table saved!
